<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 03. Clustering Jerárquico: El Árbol de las Fusiones
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 10
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/10%20-%20Clustering/Para%20Dummies/03_Clustering_Jerarquico_y_Dendrogramas_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del módulo 03 de Clustering. El cuaderno principal habló de "matrices de enlace", "métodos de linkage" y de un dataset de 64 dimensiones — aquí vamos a entender la misma idea con una analogía sencilla y con imágenes de dígitos escritos a mano, que son mucho más fáciles de "ver" que números sueltos.

Al terminar podrás explicar, con tus propias palabras:
1. En qué se diferencia el clustering jerárquico de K-Means (cuaderno 02).
2. Qué es un dendrograma y cómo "leerlo".
3. Qué es la matriz de enlace (`Z`) que arma scipy por dentro.
4. Por qué el clustering jerárquico no te obliga a decidir el número de grupos antes de empezar.


---
## 1. La analogía del torneo de parejas 💃

Imagina un salón con 100 personas bailando solas, y quieres formar grupos de baile. En vez de plantar "banderas" como en K-Means, haces lo siguiente:

1. Buscas a las **dos personas más parecidas** en todo el salón (por ejemplo, las que bailan más parecido) y las juntas en una pareja.
2. Ahora tratas a esa pareja como si fuera "una sola unidad", y vuelves a buscar las dos unidades más parecidas de todo el salón — pueden ser dos personas sueltas, o una persona suelta con la pareja que ya formaste, o dos parejas entre sí.
3. Repites el proceso una y otra vez: cada ronda, las dos unidades más parecidas se fusionan en una más grande.
4. Sigues así hasta que **todo el salón termina siendo un solo grupo gigante**.

Eso es exactamente el **clustering jerárquico aglomerativo**: en vez de arrancar con pocos grupos (como K-Means), arranca con **cada punto siendo su propio grupo** y va **fusionando** los dos grupos más parecidos, una y otra vez, hasta llegar a un único grupo con todo.

> 📌 **Para recordar:** K-Means es "de arriba hacia abajo" (defines K grupos y ajustas sus centros). El clustering jerárquico es "de abajo hacia arriba" (empiezas con muchísimos grupos chiquitos y los vas fusionando).

La gran ventaja: como registramos **el orden exacto en que se fue fusionando todo**, podemos dibujar un árbol —el **dendrograma**— y decidir *después* en qué punto "cortarlo" para quedarnos con el número de grupos que más nos convenga. No hace falta decidir K de antemano, como sí exige K-Means.


---
## Configuración del entorno de trabajo 🛠️

Vamos a trabajar con el dataset `load_digits` de scikit-learn: 1,797 imágenes pequeñas (8×8 píxeles) de dígitos escritos a mano, del 0 al 9. Cada imagen se convierte en una fila de 64 números (una intensidad de gris por píxel). Usamos el mismo dataset que el cuaderno principal porque las imágenes son mucho más intuitivas de "mirar" que puntos abstractos en un plano.

También reutilizamos el archivo de utilidades `functions/dendrogram_util.py` de este módulo (ya construido para ti), que trae dos funciones muy útiles: `plot_dendrogram` (dibuja el árbol) y `plot_node` (muestra las imágenes que quedaron agrupadas bajo una rama concreta).


In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))   # para poder importar la carpeta functions/ desde 'Para Dummies'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from scipy.cluster.hierarchy import linkage

from functions.dendrogram_util import plot_dendrogram, plot_node

digits = load_digits()
X = digits.images.reshape(-1, 8 * 8)   # cada imagen 8x8 se aplana a un vector de 64 números
y = digits.target

print("Forma de X:", X.shape, " (cada fila es una imagen de dígito, aplanada)")

fig, axes = plt.subplots(1, 10, figsize=(10, 1.4), subplot_kw={'xticks': [], 'yticks': []})
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='binary', interpolation='nearest')
    ax.set_title(str(y[i]), fontsize=9)
plt.suptitle("Primeras 10 imágenes del dataset de dígitos", y=1.15)
plt.show()

### 🤔 ¿Qué acaba de pasar?

- `load_digits()` nos entrega 1,797 imágenes pequeñas de dígitos (0-9), cada una de 8×8 píxeles.
- `digits.images.reshape(-1, 8*8)` convierte cada imagen (una cuadrícula de 8x8) en una sola fila de 64 números — así cada imagen se convierte en "un punto" con 64 características, listo para que un algoritmo de clustering lo compare con los demás.
- `y` guarda el dígito real que representa cada imagen (0 a 9), pero **no se lo mostramos al algoritmo de clustering** — lo guardamos aparte solo para poder comprobar, al final, qué tan bien se agruparon los dígitos parecidos entre sí.


---
## 2. La "bitácora de fusiones": la matriz `Z` 📒

Cuando le pedimos a scipy que arme el árbol completo (con la función `linkage`), no solo nos da un dibujo — nos da una tabla que funciona como una **bitácora**: registra, fusión por fusión, quién se juntó con quién, qué tan lejos estaban, y cuántas imágenes quedaron en el grupo resultante.

Esa tabla se llama la **matriz de enlace** (`Z`). No necesitas memorizar su formato exacto, solo entender la idea: **cada fila de `Z` es una fusión**, ordenadas de la más "barata" (imágenes casi idénticas) a la más "cara" (grupos enormes y ya bastante distintos entre sí, fusionados al final por pura necesidad).


In [ ]:
Z = linkage(X, metric='euclidean', method='ward')

print("Forma de la matriz Z:", Z.shape, f" -> {X.shape[0] - 1} fusiones registradas")
print("\nLas 5 fusiones más 'baratas' (imágenes casi idénticas):")
print(pd.DataFrame(Z[:5], columns=['grupo_1', 'grupo_2', 'distancia', 'n_imagenes']))

### 🤔 ¿Qué acaba de pasar?

- `linkage(X, metric='euclidean', method='ward')` hizo todo el proceso del torneo de parejas de la sección 1: partió de 1,797 grupos (uno por imagen) y los fue fusionando de a dos hasta quedarse con uno solo. `method='ward'` es, simplemente, la regla que usa para decidir "qué tan parecidos" son dos grupos — la veremos con más detalle más adelante.
- Cada fila de `Z` es una fusión: qué dos grupos se unieron, a qué distancia estaban, y cuántas imágenes quedan ahora en el grupo resultante.
- Las primeras filas son las fusiones "más baratas" — normalmente, dos imágenes prácticamente idénticas del mismo dígito, con una distancia muy pequeña entre ellas.


---
## 3. Dibujando el árbol: el dendrograma 🌳

Ahora sí, dibujemos el árbol completo. Como serían 1,797 fusiones (demasiadas para ver de un vistazo), lo "truncamos" para mostrar solo las últimas 100 ramas, y trazamos una línea horizontal que marca dónde tendríamos que "cortar" el árbol si quisiéramos quedarnos con 10 grupos (uno por cada dígito del 0 al 9).

Piensa en la altura del árbol como "qué tan diferentes son" los grupos que se están fusionando: fusiones muy bajas (cerca del suelo) son entre cosas casi idénticas; fusiones muy altas son entre grupos ya bastante distintos entre sí, unidos casi por obligación.


In [ ]:
fig, ax = plot_dendrogram(Z=Z, X=X, truncate_mode='lastp', p=100, n_clusters=10)
ax.set_title("Dendrograma truncado a 100 ramas (método: ward)", fontweight='bold')
ax.set_xlabel("Tamaño del grupo (o índice de la imagen)")
ax.set_ylabel("Distancia de fusión")
plt.show()

### 🤔 ¿Qué acaba de pasar?

- Cada "V" invertida del dibujo es una fusión: dos ramas que se juntan en una más grande, a la altura correspondiente a su distancia.
- La línea horizontal negra marca el punto de corte para obtener 10 grupos — imagina que trazas una tijera a esa altura: cuenta cuántas ramas verticales cruza esa línea, y ese es el número de grupos resultantes.
- Los números junto a algunos puntos de fusión son identificadores de nodo (por ejemplo `-1`, `-2`...) que podemos usar con `plot_node` para "abrir" esa rama y ver qué imágenes contiene — justo lo que haremos a continuación.


---
## 4. Abriendo una rama: `plot_node` 🔍

Inspeccionemos el **nodo `0`** — la primera fila de `Z`, es decir, la primerísima fusión que hizo el algoritmo (la más "barata" de todas, entre las dos imágenes más parecidas de todo el dataset).


In [ ]:
plot_node(Z, X, y, 0)
plt.suptitle("Nodo 0: la primera fusión (las dos imágenes más parecidas)", y=1.05)
plt.show()

### 🤔 ¿Qué acaba de pasar?

- `plot_node` recorre la rama indicada y te muestra **todas** las imágenes que terminaron agrupadas bajo ese nodo, junto con una tabla de cuántas veces aparece cada dígito real en ese grupo.
- Como pedimos el nodo `0` (la primera fusión), deberías ver solo **dos imágenes**, y casi seguro del mismo dígito — tiene sentido: es la fusión "más barata" de todas, entre las dos imágenes más parecidas de todo el dataset.
- Si en vez de `0` pidieras un nodo con índice negativo grande (por ejemplo `-14`), verías una rama mucho más tardía, con muchas más imágenes — normalmente dominada por un mismo dígito, pero ya no perfectamente "pura", porque a esa altura ya se fusionaron varios subgrupos parecidos entre sí (por ejemplo, distintos estilos de escribir el mismo número).
- Esta es una de las grandes ventajas del clustering jerárquico frente a K-Means: **no solo te da un resultado final, te deja explorar la estructura de los datos en distintos niveles de detalle**, como si fuera un árbol genealógico que puedes recorrer rama por rama.


---
## 5. ¿Qué tan "parecidos" son dos grupos? Los métodos de enlace 📏

Cuando el algoritmo decide qué dos grupos fusionar a continuación, necesita una regla para medir "qué tan lejos" están entre sí. Esa regla se llama **método de enlace** (*linkage*), y hay varias formas razonables de definirla:

* **`single`** (enlace simple): la distancia entre los dos puntos *más cercanos*, uno de cada grupo — como decir "estos dos grupos están cerca porque al menos una persona de cada uno se conoce".
* **`complete`** (enlace completo): la distancia entre los dos puntos *más lejanos* — lo opuesto, exige que **todos** se parezcan.
* **`average`**: el promedio de todas las distancias entre los puntos de un grupo y del otro.
* **`ward`**: fusiona los dos grupos que, al juntarse, generan grupos lo más "compactos" posible — es la opción por defecto y la más usada en la práctica, porque suele dar resultados equilibrados y fáciles de interpretar.

No necesitas memorizar las fórmulas — solo recuerda que **cambiar el método de enlace puede cambiar bastante la forma del árbol**, y que `ward` es un buen punto de partida por defecto.


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

modelo_jerarquico = AgglomerativeClustering(n_clusters=10, metric='euclidean', linkage='ward')
grupo_predicho = modelo_jerarquico.fit_predict(X)

print("Número de grupos formados:", len(np.unique(grupo_predicho)))
print(f"Qué tanto coinciden con los dígitos reales (Adjusted Rand Index): {adjusted_rand_score(y, grupo_predicho):.3f}")

### 🤔 ¿Qué acaba de pasar?

- `AgglomerativeClustering` hace exactamente lo mismo que `linkage`, pero te entrega directamente la etiqueta final de cada imagen (a qué grupo quedó asignada) en vez de la bitácora completa `Z` — es la opción más cómoda cuando ya no necesitas explorar el árbol y solo quieres el resultado final.
- Le pedimos `n_clusters=10` porque sabemos que hay 10 dígitos posibles — pero recuerda: en un problema real normalmente **no** conocerías ese número de antemano; aquí lo usamos solo para poder comparar contra la respuesta real.
- El **Adjusted Rand Index** compara el agrupamiento encontrado contra los dígitos reales (0 a 9): mientras más cerca de 1, mejor coincide el agrupamiento "a ciegas" con la verdad. Un valor bastante por encima de 0 nos dice que el algoritmo, sin ver ninguna etiqueta, sí logró capturar buena parte de la estructura real de los dígitos.


---
## 6. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Clustering jerárquico | Empieza con cada punto como su propio grupo y fusiona los dos más parecidos, una y otra vez, hasta quedar con uno solo. |
| Matriz de enlace (`Z`) | La "bitácora" que registra, fusión por fusión, quién se unió con quién y a qué distancia. |
| Dendrograma | El dibujo en forma de árbol de esa bitácora — se puede "cortar" a cualquier altura para obtener el número de grupos deseado. |
| `plot_node` | Herramienta para "abrir" una rama del árbol y ver qué elementos quedaron agrupados bajo ella. |
| Método de enlace (`linkage`) | La regla para medir qué tan parecidos son dos grupos (`ward`, `single`, `complete`, `average`); `ward` es la opción por defecto más estable. |
| Ventaja clave sobre K-Means | No necesitas decidir el número de grupos antes de empezar — lo decides *después*, cortando el árbol donde tenga más sentido. |

➡️ **Siguiente paso:** en el cuaderno [04 - DBSCAN (Para Dummies)](04_DBSCAN_Clustering_Dummies.ipynb) conocerás un tercer enfoque de clustering, muy distinto a los dos anteriores: en vez de centros o fusiones, agrupa según qué tan "apretados" (densos) están los puntos entre sí — y es capaz de detectar puntos raros que no pertenecen a ningún grupo.


---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
